In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from category_encoders import CountEncoder
from sklearn.tree import DecisionTreeClassifier

import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
)

from itertools import product
from tqdm import tqdm

# 1. Data preprocessing

In [2]:
df = pd.read_csv("data/training.csv")
df["PurchDate"] = pd.to_datetime(df["PurchDate"])

In [3]:
na = df.isna().sum()
na[na > 0].sort_values(ascending=False)

AUCGUART                             69564
PRIMEUNIT                            69564
WheelType                             3174
WheelTypeID                           3169
Trim                                  2360
MMRCurrentAuctionAveragePrice          315
MMRCurrentAuctionCleanPrice            315
MMRCurrentRetailAveragePrice           315
MMRCurrentRetailCleanPrice             315
MMRAcquisitionAuctionCleanPrice         18
MMRAcquisitionAuctionAveragePrice       18
MMRAcquisitionRetailAveragePrice        18
MMRAcquisitonRetailCleanPrice           18
Transmission                             9
SubModel                                 8
Color                                    8
Size                                     5
Nationality                              5
TopThreeAmericanName                     5
dtype: int64

In [4]:
q1, q2 = df["PurchDate"].quantile([1/3, 2/3])

train = df[df["PurchDate"] <= q1].copy()
valid = df[(df["PurchDate"] > q1) & (df["PurchDate"] <= q2)].copy()
test = df[df["PurchDate"] > q2].copy()

In [5]:
TARGET = "IsBadBuy"
DROP = ["RefId", "IsBadBuy", "PurchDate"]
ID_LIKE = ["WheelTypeID", "BYRNO", "VNZIP1"]   # числовые по типу, но это коды

num_cols = [c for c in df.columns
            if c not in DROP and pd.api.types.is_numeric_dtype(df[c]) and c not in ID_LIKE]
cat_cols = [c for c in df.columns
            if c not in DROP and (not pd.api.types.is_numeric_dtype(df[c]) or c in ID_LIKE)]

cat_cols_ohe = [c for c in cat_cols
                if len(df[c].unique()) < 100]
cat_cols_ce = [c for c in cat_cols
                if len(df[c].unique()) >= 100]

In [6]:
ohe_enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_enc.fit(train[cat_cols_ohe])
count_enc = CountEncoder(cols=cat_cols_ce, handle_unknown=0, handle_missing="count")
count_enc.fit(train[cat_cols_ce])

,verbose,0
,cols,"['Model', 'Trim', ...]"
,drop_invariant,False
,return_df,True
,handle_unknown,0
,handle_missing,'count'
,min_group_size,None
,combine_min_nan_groups,True
,min_group_name,None
,normalize,False


In [7]:
def make_features(part):
    X = pd.DataFrame(index=part.index)
    for c in num_cols:
        X[c] = part[c]
    ce_encoded = count_enc.transform(part[cat_cols_ce])[cat_cols_ce]
    ce_encoded.columns = [c + "_count" for c in cat_cols_ce]

    ohe_encoded = ohe_enc.transform(part[cat_cols_ohe])
    ohe_df = pd.DataFrame(ohe_encoded, columns=ohe_enc.get_feature_names_out(), index=part.index)
    return pd.concat([X, ce_encoded, ohe_df], axis=1)

def prepare(train_part, *other_parts):
    Xtr = make_features(train_part)
    med = Xtr.median()
    Xtr = Xtr.fillna(med)

    out = [Xtr]
    for part in other_parts:
        Xo = make_features(part).fillna(med)
        out.append(Xo)
    return out, list(Xtr.columns)

In [8]:
(Xtr, Xva), feat_names = prepare(train, valid)
ytr = train[TARGET].values
yva = valid[TARGET].values

# 2. Custrom Decision Tree classes

# 2.1 Decision Tree Classifier

In [9]:
from mytrees import MyNode, MyDecisionTreeClassifier

In [10]:
# tree = MyDecisionTreeClassifier(max_depth=20, n_jobs=1)
# tree.fit(pd.concat([Xtr, Xtr, Xtr, Xtr]), np.concat([ytr, ytr, ytr, ytr]))

## 2.2 Decision Tree Regressor (MSE loss)

In [11]:
from mytrees import MyNodeRegressor, MyDecisionTreeRegressor

# 3. Trying to use my DecisionTree module

In [12]:
MyTree = MyDecisionTreeClassifier(max_depth=10, n_jobs=2)
MyTree.fit(Xtr, ytr)

In [13]:
def gini_score(y_true, y_proba):
    return roc_auc_score(y_true, y_proba) * 2 -1

In [14]:
predictions = MyTree.predict_proba(Xva)
gini = gini_score(yva, predictions[:, 1])
print(gini)

0.3705636810307613


# 4. Trying to use sklearn's DecisionTree classifier

In [15]:
SkTree = DecisionTreeClassifier(max_depth=10)
SkTree.fit(Xtr, ytr)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

In [16]:
predictions = SkTree.predict_proba(Xva)
gini = gini_score(yva, predictions[:, 1])
print(gini)

0.3576103513918343


Sklearn's implementation of the decision tree is faster because the code is optimized in C, but it shows slightly less gini score.

The slight difference in gini may be due to a different algorithm for selecting the partition threshold in my and sklearns implementation of DecisionTreeClassifier.

# 5. Implementation of RandomForestClassifier

In [17]:
from mytrees import MyDecisionTreeClassifier, MyRandomForestClassifier

In [18]:
MyForest = MyRandomForestClassifier(max_depth=6, n_estimators=100, n_jobs=12, seed=42)
MyForest.fit(Xtr, ytr)

In [19]:
predictions = MyForest.predict_proba(Xva)
gini = gini_score(yva, predictions[:, 1])
print(gini)

0.4543473215797733


# 6. Implementation of GBDT classifier

In [20]:
from mytrees import MyGradientBoostingClassifier

In [21]:
clf = MyGradientBoostingClassifier(max_depth=7, number_of_trees=15, max_features=Xtr.shape[1])
clf.fit(Xtr, ytr)

In [22]:
predictions = clf.predict_proba(Xva)
gini = gini_score(yva, predictions)
print(gini)

0.44604242111940273


# 7. LightGBM, Catboost and XGBoost

# 7.1 XGBoost

Разложение функции потерь в ряд Тейлора до второго порядка:

$$
L(F(x_i) + w) \approx L(F(x_i)) + g_i \cdot w + \frac{1}{2} \cdot h_i \cdot w^2
$$
где:
- $g_i$ — градиент (первая производная) потерь по $F(x_i)$
- $h_i$ — гессиан (вторая производная) потерь по $F(x_i)$

Суммарная целевая функция с регуляризацией:
$$
\text{Obj}(w) = \left( \sum g_i \right) \cdot w + \frac{1}{2} \cdot \left( \sum h_i + \lambda \right) \cdot w^2
$$
где $\lambda$ — параметр регуляризации (штраф за сложность).
Минимум квадратичной функции достигается в точке:
$$
w = -\frac{\sum g_i}{\sum h_i + \lambda}
$$

Особенности:
- Гистограммный алгоритм: непрерывные признаки заранее дискретизируются в 255 бинов; поиск сплита идёт по гистограммам, а не по отсортированным значениям.
- Рост дерева level-wise (по умолчанию): дерево наращивается уровень за уровнем до max_depth, что даёт сбалансированные деревья.
- Sparsity-aware split: для каждого узла обучается «направление по умолчанию» для пропусков. NaN не нужно заполнять, алгоритм сам решает, куда их отправлять.
- Категориальные признаки нативно практически не поддерживаются

**DART (Dropouts meet Multiple Additive Regression Trees).** - это режим бустера (booster="dart"), переносящий идею dropout из нейросетей в бустинг. Проблема обычного бустинга: первые деревья вносят основной вклад, а поздние лишь «подчищают» ошибки, из-за чего ансамбль переобучается. DART решает это следующим образом.

На каждой итерации с определенной вероятностью каждое дерево временно выбрасывается из ансамбля; градиенты считаются по «урезанному» ансамблю, и новое дерево исправляет именно его ошибки. Затем, чтобы суммарный вклад не «перепрыгнул» цель (новое дерево дублирует вклад выброшенных), веса нормализуются: если выброшено k деревьев, новое дерево умножается на 1/(k+1), а выброшенные — на k/(k+1). В итоге поздние деревья вносят более равномерный вклад, переобучение снижается, но обучение медленнее и предсказание требует всего ансамбля (нельзя рано остановиться так же просто, как в обычном режиме).

In [23]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
base_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

param_grid = {
    "n_estimators": [200, 500, 1000],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [4, 6],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}
 
grid = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring="roc_auc",     
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True,               
)
 
grid.fit(Xtr, ytr)
print("Лучшие параметры:", grid.best_params_)
print("Лучший CV score:", grid.best_score_)
bst = grid.best_estimator_

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Лучшие параметры: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 500, 'subsample': 0.8}
Лучший CV score: 0.7631209970260976


In [24]:
best_params = {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 500, 'subsample': 0.8}
XGBbest_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    **best_params
)
XGBbest_model.fit(Xtr, ytr)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [25]:
preds = XGBbest_model.predict_proba(Xva)[:, 1]
gini = gini_score(yva, preds)
print(gini)

0.4912736985780506


# 7.2 LightGBM
Ключевая особенность - скорость и масштабируемость без потери качества. Также обучается разложением в ряд Тейлора до второго порядка, нодерево растёт leaf-wise: на каждом шаге делится тот лист, который даёт максимальное уменьшение потерь, где бы он ни находился. Это даёт глубокие несимметричные деревья, которые быстрее снижают ошибку при том же числе листьев, но легче переобучаются. 

**Поддерживает DART.**

Основные механизмы ускорения алгоритма:
- Гистограммный алгоритм: непрерывные признаки заранее дискретизируются в 255 бинов; поиск сплита идёт по гистограммам, а не по отсортированным значениям.
- EFB (Exclusive Feature Bundling): взаимоисключающие разреженные признаки (например, после one-hot, где не бывает двух ненулевых одновременно) склеиваются в один, что сокращает число признаков.
- Категориальные признаки поддерживаются нативно: категории сортируются по статистике градиентов ($\frac{\sum g_i}{\sum h_i + cat\_smooth}$), и ищется оптимальное разбиение множества категорий на две группы.
- GOSS (Gradient-based One-Side Sampling): объекты с большими градиентами (плохо предсказанные) сохраняются все, а из хорошо предсказанных берётся случайная подвыборка с компенсирующим весом. <img src="misc\images\efb_feature_bundling.svg" width="50%">



In [26]:
XtrLGB = train.drop(columns=DROP) 
XtrLGB[cat_cols]  = XtrLGB[cat_cols].fillna("MISSING").astype("category")
XvaLGB = valid.drop(columns=DROP) 
XvaLGB[cat_cols]  = XvaLGB[cat_cols].fillna("MISSING").astype("category")

linear_tree = True
train_data = lgb.Dataset(XtrLGB, label=ytr, params={'linear_tree': linear_tree})

In [27]:
grid = {
    'num_leaves': [15, 31, 63],
    'learning_rate': [0.03, 0.1],
    'feature_fraction': [0.8, 1.0],
    'n_round': [100, 200, 500]
}


best = None
combos = list(product(*grid.values()))   
pbar = tqdm(combos, desc="grid search")
for nl, lr, ff, nr in pbar:
    p = {
        'objective': 'binary', 
        'metric': 'auc', 
        'verbose': -1,
        'num_leaves': nl,
        'num_threads': -1,
        'learning_rate': lr, 
        'feature_fraction': ff,
        'linear_tree': linear_tree
    }
    res = lgb.cv(p, train_data, num_boost_round=nr, nfold=5,
                 stratified=True, seed=42)
    score = res['valid auc-mean'][-1]
    n_iter = len(res['valid auc-mean'])
    if best is None or score > best[0]:
        best = (score, n_iter, p)

grid search: 100%|██████████| 36/36 [07:02<00:00, 11.72s/it]


In [28]:
print(best)

(0.752118838799171, 100, {'objective': 'binary', 'metric': 'auc', 'verbose': -1, 'num_leaves': 15, 'num_threads': -1, 'learning_rate': 0.03, 'feature_fraction': 0.8, 'linear_tree': True})


In [29]:
LightGBM_model = lgb.train(best[2], train_data)

In [30]:
preds = LightGBM_model.predict(XvaLGB)
gini = gini_score(yva, preds)
print(gini)

0.45439619712822443


# 7.3 CatBoost

CatBoost (Яндекс, 2017) сфокусирован на двух проблемах: категориальные признаки и скрытая утечка целевой переменной в классическом бустинге.

**CatBoost строит симметричные деревья:** на каждом уровне дерева во всех узлах используется один и тот же признак и порог. Это ускоряет инференс, потому что для классификации нескольких объектов требуется только 1 раз прочитать массив и ко всем объектам задать одинаковые вопросы. Также это работает как регуляризация, поскольку структура деревьев сильно упрощается.

**Ordered boosting:** в классическом бустинге градиент для объекта считается по модели, которая уже видела этот объект при обучении, возникает утечка данных и смещение. CatBoost генерирует случайные перестановки данных и для объекта i использует модель, обученную только на объектах, стоящих в перестановке до него. На небольших выборках это заметно снижает переобучение.

**Встроенная обработка категориальных признаков:** CatBoost не делает one-hot encoding (кроме признаков с малым числом категорий), а заменяет категорию упорядоченной целевой статистикой.
1. Данные выстраиваются в случайную перестановку (на разных итерациях используются разные перестановки, чтобы первые объекты не страдали от бедной статистики).
2. Для объекта i значение категории c заменяется на сглаженное среднее таргета: (сумма y по объектам с категорией c среди предыдущих + a·p) / (число таких объектов + a), где p — априорное среднее (prior), a — вес сглаживания. Сглаживание защищает редкие категории от шумных оценок.
3. Дополнительно CatBoost на лету строит комбинации категориальных признаков (например, «город × категория товара»)

<img src="misc\images\catboost_ordered_target_encoding.svg" width="50%">


In [31]:
XtrCatBoost = train.drop(columns=DROP) 
XtrCatBoost[cat_cols]  = XtrCatBoost[cat_cols].fillna("MISSING").astype(str)
XvaCatBoost = valid.drop(columns=DROP) 
XvaCatBoost[cat_cols]  = XvaCatBoost[cat_cols].fillna("MISSING").astype(str)


In [32]:
CatBoost_model = CatBoostClassifier()
CatBoost_model.fit(X=XtrCatBoost, y=ytr, cat_features=cat_cols)

Learning rate set to 0.040298
0:	learn: 0.6562043	total: 240ms	remaining: 3m 59s
1:	learn: 0.6231712	total: 310ms	remaining: 2m 34s
2:	learn: 0.5936029	total: 382ms	remaining: 2m 6s
3:	learn: 0.5671949	total: 428ms	remaining: 1m 46s
4:	learn: 0.5412629	total: 495ms	remaining: 1m 38s
5:	learn: 0.5187777	total: 566ms	remaining: 1m 33s
6:	learn: 0.4977346	total: 635ms	remaining: 1m 30s
7:	learn: 0.4789782	total: 702ms	remaining: 1m 27s
8:	learn: 0.4630650	total: 773ms	remaining: 1m 25s
9:	learn: 0.4488634	total: 830ms	remaining: 1m 22s
10:	learn: 0.4347347	total: 893ms	remaining: 1m 20s
11:	learn: 0.4221955	total: 989ms	remaining: 1m 21s
12:	learn: 0.4116755	total: 1.05s	remaining: 1m 19s
13:	learn: 0.4013598	total: 1.13s	remaining: 1m 19s
14:	learn: 0.3922054	total: 1.18s	remaining: 1m 17s
15:	learn: 0.3841003	total: 1.26s	remaining: 1m 17s
16:	learn: 0.3773530	total: 1.31s	remaining: 1m 15s
17:	learn: 0.3701235	total: 1.39s	remaining: 1m 15s
18:	learn: 0.3634249	total: 1.47s	remaining: 

CatBoostClassifier()

In [33]:
preds = CatBoost_model.predict_proba(XvaCatBoost)[:, 1]
gini = gini_score(yva, preds)
print(gini)

0.48876363787744626


# 8. Evaluation of the best model

In [34]:
(Xtr_, Xts), _ = prepare(train, test)
yts = test[TARGET].values

In [35]:
preds_test = XGBbest_model.predict_proba(Xts)[:, 1]
gini = gini_score(yts, preds_test)
print(gini)

0.5006491753814692


# 9*. Bonus task

In [36]:
ETC_classifier = MyRandomForestClassifier(max_depth=6, n_estimators=100, n_jobs=4, ETC=True)
ETC_classifier.fit(Xtr, ytr)

In [37]:
predictions = ETC_classifier.predict_proba(Xva)
gini = gini_score(yva, predictions[:, 1])
print(gini)

0.4714028794051113
